# AMP and the Limits of PyTorch Profiler
---

We fixed the DataLoader bottleneck and got a 2.8× speedup.  Now let's push further.  Automatic Mixed Precision (AMP) is one of the highest-leverage optimisations available for GPU training — it often delivers a 1.5–2× speedup with essentially no code changes and no loss in model quality.  In this notebook we will add AMP to our training loop, discover that it is not performing as expected, and reach the limits of what PyTorch Profiler can tell us.

## Baseline Reminder

Before making any changes, let's confirm our starting point.  Run the fixed v1 script to get a fresh timing measurement.

In [ ]:
!python ../source_code/intro/train_v1_fixed.py

**Expected output:**

```
steps timed: 55  mean step: ~94 ms  throughput: ~2727 img/s
```

This is our FP32 baseline with a healthy DataLoader.  Keep this number in mind — AMP should roughly double this throughput.

## What Is AMP?

Modern NVIDIA GPUs have dedicated Tensor Core hardware that executes matrix multiplications in lower-precision formats significantly faster than in FP32.  Automatic Mixed Precision lets you take advantage of this without rewriting your model.

On modern hardware (Ampere, Ada Lovelace, Hopper, Blackwell — including the L4s we are running on) the standard choice is **bfloat16**.  It has the same dynamic range as FP32 but only 7 mantissa bits instead of 23, so it captures the speedups of half-precision without the underflow problems FP16 historically caused for gradient computations.

`torch.amp.autocast` is a context manager that automatically casts eligible operations (convolutions, matrix multiplications, attention) to bf16 while keeping numerically sensitive operations (loss, batch norm, softmax) in FP32:

```python
with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
    logits = model(x)      # conv/matmul run in bf16
    loss = loss_fn(logits, y)  # loss computed in FP32
```

That is the whole change.  Because bf16 has FP32's dynamic range, gradients do not underflow, so backward and the optimizer step look exactly like they did in FP32 — no gradient scaler needed.  (Loss scaling with `torch.amp.GradScaler` is still important for FP16 on older hardware or memory-constrained workloads where the precision tradeoff is worth it; it is just outside the scope of this workshop.)

## The Code Changes: [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) → [`train_v2.py`](../source_code/intro/train_v2.py)

Here is the diff between the fixed FP32 loop and the new AMP loop:

```python
# Forward pass now wrapped in autocast:
with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):   # ← new
    logits = model(x)
    loss = loss_fn(logits, y)

# Backward and optimizer step are unchanged — bf16 doesn't need a scaler:
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

One change, no model architecture changes, no gradient scaler.  Expected result: ~2× speedup over the FP32 baseline.

## Run It — Is It 2× Faster?

In [ ]:
!python ../source_code/intro/train_v2.py

**Expected output:**

```
steps timed: 55  mean step: ~66 ms  throughput: ~3880 img/s
```

| Script | Mean step | Throughput | vs FP32 baseline |
|---|---|---|---|
| [`train_v1_fixed.py`](../source_code/intro/train_v1_fixed.py) (FP32) | ~94 ms | ~2727 img/s | — |
| [`train_v2.py`](../source_code/intro/train_v2.py) (AMP) | ~66 ms | ~3880 img/s | **~1.4×** |

We got a speedup, but only ~1.4× — not the ~2× we would expect from AMP on this GPU.  Something is eating nearly half of our potential gain.

Time to profile.

## Profile the AMP Run

[`train_v2_profile.py`](../source_code/intro/train_v2_profile.py) wraps the same AMP training loop with `torch.profiler`.  Let's collect a trace.

In [ ]:
!python ../source_code/intro/train_v2_profile.py

In [ ]:
from IPython.display import display, Javascript
display(Javascript("""
  var tb_url = 'https://' + window.location.hostname.replace('notebooks-', 'tensorboard-');
  element.innerHTML = '<b>TensorBoard:</b> <a href="' + tb_url + '" target="_blank">' + tb_url + '</a>';
"""))

**Expected output:**

```
steps timed: 55  mean step: ~72 ms  throughput: ~3555 img/s
Trace written to /workspace/logs/train_v2_profile
```

Open TensorBoard (your link is shown in the cell above) and select the `train_v2_profile` run.

Compare the **Overview** step breakdowns side by side:

```
train_v1_fixed  (FP32, healthy):
  DataLoader  ██  ~8 ms
  Forward     ████  ~15 ms
  Backward    ████████  ~28 ms
  Optimizer   ██  ~6 ms
  ─────────────────────────────  ~57 ms GPU-active

train_v2_profile  (AMP + something wrong):
  DataLoader  ██  ~8 ms
  Forward     ██  ~10 ms      ← AMP forward is faster ✓
  Backward    ████████████████  ~48 ms   ← much slower than FP32!
  Optimizer   ██  ~6 ms
```

The forward pass *did* get faster — AMP is working for forward.  But the backward pass is **dramatically slower** than even our FP32 baseline.  The profiler is telling us:

> **The backward section is the problem.**

But that is all it can tell us.  The backward pass is just... slow.  The profiler shows us the duration, but not why the GPU is spending so long there.  Is it the gradient computation?  Some interaction AMP introduced?  Something else entirely?

Look at the **Operator** view in TensorBoard.  You might notice some unusual operations in the backward timeline, but it is hard to tell which ones are the cause and which are the symptom.

## The Limit of the Step-Level View

PyTorch Profiler's step view answers **where**: the backward section.

It cannot answer **why**: what is happening *inside* the backward pass that is making it 70 % slower than it should be?

To answer that, we need to see the GPU timeline — a continuous record of which kernels are running, when the GPU is idle, and what the CPU is doing in between.  That is what NVIDIA Nsight Systems provides.

In the next notebook we will run the same code under `nsys` and open the GPU timeline.  What we find there will make the problem immediately obvious.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red; height:80px;"><b><br/>[Next Notebook — Finding the Root Cause with Nsight Systems](intro-nsys.ipynb)</b></div></center>

---

## Links and Resources

- [PyTorch Automatic Mixed Precision documentation](https://pytorch.org/docs/stable/amp.html)
- [AMP recipe](https://pytorch.org/tutorials/recipes/recipes/amp_recipe.html)
- [PyTorch Profiler documentation](https://pytorch.org/docs/stable/profiler.html)

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0).